# Complete Computer Vision Workflow
## Deception Detection from Video Frames

This notebook walks through a **full image-classification pipeline** for detecting
deception from facial video frames extracted from the **DOLOS dataset** (ICCV 2023).

### Pipeline Overview
1. Import libraries
2. Load / prepare dataset
3. Exploratory data analysis (EDA)
4. Data cleaning
5. Image enhancement
6. Color-space processing (RGB, grayscale, HSV)
7. Resizing
8. Normalization
9. Data augmentation
10. Train / test split
11. Build CNN
12. Compile model
13. Train model
14. Evaluate model
15. Visualize results
16. Save model
17. Predict on new images

---



---
## Step 1: Import Libraries

We import all required libraries for data processing, computer vision,
deep learning, and visualization.



In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import time
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_curve, auc,
    precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0, 'font.size': 11})

ROOT = r'C:\Users\OMEN\Desktop\DECEPTION DETECTION USING VIDEOS AND FACE'
OUTPUT_DIR = os.path.join(ROOT, 'cv_output')
MODEL_DIR = os.path.join(ROOT, 'saved_models')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print("All imports successful.")



---
## Step 2: Load / Prepare Dataset

### Data Source
- **DOLOS Dataset**: Deception clips from the British TV gameshow *Would I Lie to You?*
- **Videos**: 1427 `.mp4` clips in `DOLOSDATA/raw_videos/`
- **Metadata**: `dolos_timestamps.csv` with file names, labels, timestamps

### Preparation Strategy
1. Read the metadata CSV to get clip names and labels
2. For each video, extract **10 evenly-spaced frames** using OpenCV
3. Store frames as numpy arrays
4. Build arrays mapping each frame to its label

This converts the video problem into an **image classification** problem.



In [ ]:
CSV_PATH = r'C:\Users\OMEN\Desktop\DECEPTION DETECTION USING VIDEOS AND FACE\DOLOSDATA\DOLOS\dolos_timestamps.csv'
VIDEO_DIR = r'C:\Users\OMEN\Desktop\DECEPTION DETECTION USING VIDEOS AND FACE\DOLOSDATA\raw_videos'

df_meta = pd.read_csv(CSV_PATH)
print(f"Metadata shape: {df_meta.shape}")
print(f"\nLabel distribution:\n{df_meta['label'].value_counts()}")
print(f"\nSample rows:")
df_meta.head(10)



In [ ]:
video_files = set(f.replace('.mp4', '') for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4'))
df_meta['video_available'] = df_meta['file_name'].isin(video_files)
print(f"Videos on disk: {df_meta['video_available'].sum()} / {len(df_meta)}")

df_meta['label_clean'] = df_meta['label'].str.lower().str.strip()
df_meta['label_clean'] = df_meta['label_clean'].replace({
    'lie': 'deception', 'true': 'truth'
})
print(f"\nClean label distribution:\n{df_meta['label_clean'].value_counts()}")



### Frame Extraction from Videos

We extract **10 evenly-spaced frames** from each video clip.
DOLOS videos are face close-ups, so we skip face detection and resize directly.



In [ ]:
FRAMES_PER_VIDEO = 10
TARGET_SIZE = (128, 128)

def extract_frames(video_path, n_frames=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return []

    frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        face_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face_rgb = cv2.resize(face_rgb, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        frames.append(face_rgb)

    cap.release()
    return frames

print("Frame extraction function defined. (DOLOS videos are face close-ups, skipping face detection)")



In [ ]:
available_df = df_meta[df_meta['video_available']].drop_duplicates(subset='file_name').copy()
print(f"Total videos available on disk: {len(available_df)}")
print(f"Processing ALL videos...\n")

all_frames = []
all_labels = []
all_filenames = []

for i, (_, row) in enumerate(available_df.iterrows()):
    video_path = os.path.join(VIDEO_DIR, row['file_name'] + '.mp4')
    frames = extract_frames(video_path)

    for frame in frames:
        all_frames.append(frame)
        all_labels.append(row['label_clean'])
        all_filenames.append(row['file_name'])

    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(available_df)} videos...")

all_frames = np.stack(all_frames, axis=0)
all_labels = np.array(all_labels)

print(f"\nDone! Extracted {len(all_frames)} frames from {len(available_df)} videos")
print(f"Frame array shape: {all_frames.shape}")
print(f"Label distribution: {dict(zip(*np.unique(all_labels, return_counts=True)))}")



---
## Step 3: Exploratory Data Analysis (EDA)

We visualize the extracted frames and understand the data distribution.



In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Sample Extracted Frames", fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < len(all_frames):
        ax.imshow(all_frames[i])
        ax.set_title(f"Label: {all_labels[i]}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

label_counts = pd.Series(all_labels).value_counts()
colors = {'truth': '#2ecc71', 'deception': '#e74c3c'}
axes[0].bar(label_counts.index, label_counts.values,
            color=[colors.get(l, '#999') for l in label_counts.index],
            edgecolor='black', linewidth=1.2)
for i, (l, v) in enumerate(label_counts.items()):
    axes[0].text(i, v + 2, f'{v} ({v/len(all_labels)*100:.1f}%)',
                ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Frame Count by Label', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')

shapes = [f.shape for f in all_frames]
heights = [s[0] for s in shapes]
widths = [s[1] for s in shapes]
axes[1].hist(heights, bins=30, color='#3498db', edgecolor='black', alpha=0.7, label='Height')
axes[1].hist(widths, bins=30, color='#e67e22', edgecolor='black', alpha=0.7, label='Width')
axes[1].set_title('Frame Dimensions Distribution', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Pixels')
axes[1].set_ylabel('Count')
axes[1].legend()

brightness = [np.mean(f) for f in all_frames]
df_brightness = pd.DataFrame({'brightness': brightness, 'label': all_labels})
sns.boxplot(x='label', y='brightness', data=df_brightness, ax=axes[2],
            palette={'truth': '#2ecc71', 'deception': '#e74c3c'})
axes[2].set_title('Mean Brightness by Label', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()



---
## Step 4: Data Cleaning

We perform the following cleaning steps:
1. Remove corrupted or unreadable frames
2. Remove frames that are too small (< 32x32)
3. Remove frames with very low variance (blank/black frames)
4. Remove duplicates (identical pixel arrays)



In [ ]:
def clean_frames(frames, labels, filenames):
    clean_idx = []
    seen_hashes = set()

    for i, (frame, label) in enumerate(zip(frames, labels)):
        if frame.shape[0] < 32 or frame.shape[1] < 32:
            continue
        if np.var(frame) < 50:
            continue
        frame_hash = hash(frame.tobytes())
        if frame_hash in seen_hashes:
            continue
        seen_hashes.add(frame_hash)
        clean_idx.append(i)

    clean_frames = frames[clean_idx]
    clean_labels = labels[clean_idx]
    clean_filenames = np.array(filenames)[clean_idx]

    removed = len(frames) - len(clean_frames)
    print(f"Removed {removed} / {len(frames)} frames ({removed/len(frames)*100:.1f}%)")
    print(f"Remaining: {len(clean_frames)} frames")
    print(f"Label distribution: {dict(zip(*np.unique(clean_labels, return_counts=True)))}")

    return clean_frames, clean_labels, clean_filenames

all_frames, all_labels, all_filenames = clean_frames(all_frames, all_labels, all_filenames)



---
## Step 5: Image Enhancement

We apply various image enhancement techniques to improve frame quality:

1. **Histogram Equalization (CLAHE)**: Enhances contrast in lighting-varied frames
2. **Gaussian Blur Denoising**: Removes noise while preserving edges
3. **Sharpening**: Enhances facial feature edges
4. **Brightness/Contrast Adjustment**: Normalizes illumination differences

These enhancements help the CNN focus on facial deception cues rather than
recording quality artifacts.



In [ ]:
def apply_clahe(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def apply_denoise(frame):
    return cv2.GaussianBlur(frame, (3, 3), 0)

def apply_sharpen(frame):
    blurred = cv2.GaussianBlur(frame, (0, 0), 3)
    return cv2.addWeighted(frame, 1.5, blurred, -0.5, 0)

def adjust_brightness_contrast(frame, brightness=1.0, contrast=1.0):
    frame = frame.astype(np.float32)
    frame = frame * contrast + (brightness - 1.0) * 128
    frame = np.clip(frame, 0, 255).astype(np.uint8)
    return frame

sample_frame = all_frames[0].copy()

enhanced_frames = {
    'Original': sample_frame,
    'CLAHE': apply_clahe(sample_frame),
    'Denoised': apply_denoise(sample_frame),
    'Sharpened': apply_sharpen(sample_frame),
    'Bright+Contrast': adjust_brightness_contrast(sample_frame, 1.2, 1.1),
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle("Image Enhancement Techniques", fontsize=14, fontweight='bold')
for ax, (name, frame) in zip(axes, enhanced_frames.items()):
    ax.imshow(frame)
    ax.set_title(name, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
print("Applying CLAHE enhancement to all frames...")
enhanced_frames = np.array([apply_clahe(f) for f in all_frames])
print(f"Enhancement complete. Shape: {enhanced_frames.shape}")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Before vs After CLAHE Enhancement", fontsize=14, fontweight='bold')
for i in range(5):
    axes[0, i].imshow(all_frames[i])
    axes[0, i].set_title(f"Original\n{all_labels[i]}", fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(enhanced_frames[i])
    axes[1, i].set_title(f"CLAHE\n{all_labels[i]}", fontsize=9)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()



---
## Step 6: Color-Space Processing

We convert frames to multiple color spaces to create different input representations:

1. **RGB**: Standard color representation (3 channels)
2. **Grayscale**: Intensity only (1 channel) - removes color bias
3. **HSV**: Hue-Saturation-Value - separates color from intensity, useful for skin tone analysis
4. **Edge Maps**: Canny edge detection - captures facial contours and micro-expressions

Each representation captures different aspects of facial deception cues.



In [ ]:
def to_grayscale(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

def to_hsv(frame):
    return cv2.cvtColor(frame, cv2.COLOR_RGB2HSV)

def to_edges(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

sample = all_frames[0].copy()
color_spaces = {
    'RGB (Original)': sample,
    'Grayscale': to_grayscale(sample),
    'HSV': to_hsv(sample),
    'Edge Map': to_edges(sample),
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
fig.suptitle("Color-Space Conversions", fontsize=14, fontweight='bold')
for ax, (name, frame) in zip(axes, color_spaces.items()):
    cmap_val = 'gray' if 'Grayscale' in name or 'Edge' in name else None
    ax.imshow(frame, cmap=cmap_val)
    ax.set_title(name, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
print("Preparing color-space variants...")

rgb_frames = enhanced_frames.copy()
gray_frames = np.array([to_grayscale(f) for f in enhanced_frames])
hsv_frames = np.array([to_hsv(f) for f in enhanced_frames])
edge_frames = np.array([to_edges(f) for f in enhanced_frames])

print(f"RGB frames shape:  {rgb_frames.shape}")
print(f"Gray frames shape: {gray_frames.shape}")
print(f"HSV frames shape:  {hsv_frames.shape}")
print(f"Edge frames shape: {edge_frames.shape}")



---
## Step 7: Resizing

All frames are resized to a **uniform 128x128 pixels** to ensure consistent
input dimensions for the CNN. This is a common resolution that balances
detail preservation with computational efficiency.



In [ ]:
TARGET_SIZE = (128, 128)

def resize_frames(frames, target_size=TARGET_SIZE):
    resized = np.array([cv2.resize(f, target_size, interpolation=cv2.INTER_AREA)
                        for f in frames])
    return resized

rgb_resized = resize_frames(rgb_frames)
gray_resized = resize_frames(gray_frames)
hsv_resized = resize_frames(hsv_frames)
edge_resized = resize_frames(edge_frames)

print(f"Resized to: {TARGET_SIZE}")
print(f"RGB shape:  {rgb_resized.shape}")
print(f"Gray shape: {gray_resized.shape}")
print(f"HSV shape:  {hsv_resized.shape}")
print(f"Edge shape: {edge_resized.shape}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("After Resizing to 128x128", fontsize=14, fontweight='bold')
for ax, (name, frame) in zip(axes, [
    ('RGB', rgb_resized[0]),
    ('Grayscale', gray_resized[0]),
    ('HSV', hsv_resized[0]),
    ('Edges', edge_resized[0])
]):
    cmap_val = 'gray' if 'Gray' in name or 'Edge' in name else None
    ax.imshow(frame, cmap=cmap_val)
    ax.set_title(f"{name}\n{frame.shape}", fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()



---
## Step 8: Normalization

We normalize pixel values to the **[0, 1]** range by dividing by 255.
This is essential for stable CNN training -- raw pixel values (0-255) cause
large gradient magnitudes that destabilize optimization.



In [ ]:
rgb_normalized = rgb_resized.astype(np.float32) / 255.0
gray_normalized = gray_resized.astype(np.float32) / 255.0
hsv_normalized = hsv_resized.astype(np.float32) / 255.0
edge_normalized = edge_resized.astype(np.float32) / 255.0

label_map = {'truth': 0, 'deception': 1}
encoded_labels = np.array([label_map[l] for l in all_labels])

print(f"RGB pixel range:  [{rgb_normalized.min():.2f}, {rgb_normalized.max():.2f}]")
print(f"Gray pixel range: [{gray_normalized.min():.2f}, {gray_normalized.max():.2f}]")
print(f"Label encoding: truth=0, deception=1")
print(f"Label distribution: {dict(zip(*np.unique(encoded_labels, return_counts=True)))}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(rgb_resized[0].flatten(), bins=50, color='#e74c3c', edgecolor='black')
axes[0].set_title('Before Normalization (0-255)', fontweight='bold')
axes[0].set_xlabel('Pixel Value')
axes[1].hist(rgb_normalized[0].flatten(), bins=50, color='#2ecc71', edgecolor='black')
axes[1].set_title('After Normalization (0-1)', fontweight='bold')
axes[1].set_xlabel('Pixel Value')
plt.tight_layout()
plt.show()



---
## Step 9: Data Augmentation

Since deception datasets are typically small, we apply **data augmentation**
to artificially increase training set diversity. Augmentations simulate
real-world variations:

1. **Horizontal flip**: Mirror symmetry (face identity preserved)
2. **Rotation**: +/-15 degrees
3. **Brightness shift**: +/-20%
4. **Zoom**: +/-20%

We use `torchvision.transforms` for on-the-fly augmentation during training.
Augmentation is applied **only to training data** -- test data remains clean.



In [ ]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    T.ToTensor(),
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
])

# Demo: show augmentation effects
sample_img = (rgb_normalized[0] * 255).astype(np.uint8)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Data Augmentation Examples", fontsize=14, fontweight='bold')
axes[0, 0].imshow(rgb_normalized[0])
axes[0, 0].set_title("Original", fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    aug_img = train_transform(sample_img).permute(1, 2, 0).numpy()
    row, col = divmod(i, 5)
    axes[row, col].imshow(np.clip(aug_img, 0, 1))
    axes[row, col].set_title(f"Augmented {i}", fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()



---
## Step 10: Train / Test Split

We split the data into:
- **Training set (70%)**: For model learning
- **Validation set (15%)**: For hyperparameter tuning and early stopping
- **Test set (15%)**: For final unbiased evaluation

We use **stratified splitting** to maintain class balance across all sets.



In [ ]:
X = rgb_normalized
y = encoded_labels

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nTrain label dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val label dist:   {dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"Test label dist:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Dataset Split Distribution", fontsize=14, fontweight='bold')

for ax, (name, labels) in zip(axes, [
    ('Train', y_train), ('Validation', y_val), ('Test', y_test)
]):
    counts = pd.Series(labels).map({0: 'Truth', 1: 'Deception'}).value_counts()
    colors_list = ['#2ecc71', '#e74c3c']
    ax.bar(counts.index, counts.values, color=colors_list, edgecolor='black')
    ax.set_title(f'{name} (n={len(labels)})', fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
# Convert to PyTorch tensors and create DataLoaders
# Transpose from (N, H, W, C) to (N, C, H, W) for PyTorch
X_train_t = torch.tensor(X_train.transpose(0, 3, 1, 2), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(X_val.transpose(0, 3, 1, 2), dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.long)
X_test_t  = torch.tensor(X_test.transpose(0, 3, 1, 2), dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t, y_val_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print(f"Input shape:   {X_train_t.shape}")



---
## Step 11: Build CNN

We build a **Custom CNN** architecture for binary image classification using PyTorch:

### Architecture
```
Input (3, 128, 128)
+-- Conv2D(32, 3x3) + BatchNorm + ReLU + MaxPool
+-- Conv2D(64, 3x3) + BatchNorm + ReLU + MaxPool
+-- Conv2D(128, 3x3) + BatchNorm + ReLU + MaxPool
+-- Conv2D(256, 3x3) + BatchNorm + ReLU + MaxPool
+-- AdaptiveAvgPool2d(1)
+-- Flatten
+-- Linear(256) + BatchNorm + Dropout(0.5) + ReLU
+-- Linear(128) + BatchNorm + Dropout(0.3) + ReLU
+-- Linear(2) (logits output)
```

### Design Choices
- **Batch Normalization**: Stabilizes training, allows higher learning rates
- **AdaptiveAvgPool**: Handles variable spatial sizes, reduces parameters
- **Dropout**: Regularization to prevent overfitting on small dataset
- **CrossEntropyLoss**: Combines LogSoftmax + NLLLoss (standard for classification)



In [ ]:
class DeceptionCNN(nn.Module):
    def __init__(self, input_channels=3, num_classes=2):
        super(DeceptionCNN, self).__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # Block 4
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x

model = DeceptionCNN(input_channels=3, num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: DeceptionCNN")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)



---
## Step 12: Compile Model

We configure the model for training in PyTorch:

- **Optimizer**: Adam (adaptive learning rate, fast convergence)
- **Loss**: CrossEntropyLoss (standard for classification)
- **Scheduler**: ReduceLROnPlateau (halve LR if validation loss plateaus)
- **Early Stopping**: Stop if validation loss doesn't improve for 10 epochs



In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model compiled with:")
print("  Optimizer: Adam (lr=1e-3, weight_decay=1e-4)")
print("  Loss: CrossEntropyLoss")
print("  Scheduler: ReduceLROnPlateau")
print(f"  Device: {device}")



---
## Step 13: Train Model

We train the CNN for up to **50 epochs** with early stopping.
The training loop includes:
- Forward pass on training batches
- Backward pass with gradient computation
- Validation after each epoch
- Learning rate scheduling
- Early stopping based on validation loss



In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)

print("Training functions defined.")



In [ ]:
EPOCHS = 50
patience = 10
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f"Training for up to {EPOCHS} epochs...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")
print("=" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels, val_probs = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:2d}/{EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}  "
          f"LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, os.path.join(MODEL_DIR, 'best_cnn_model.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model (val_loss: {best_val_loss:.4f})")

print(f"Training completed after {len(history['train_loss'])} epochs")



---
## Step 14: Evaluate Model

We evaluate the trained model on the **test set** using multiple metrics:
- Accuracy
- Precision & Recall
- F1 Score
- AUC-ROC
- Confusion Matrix
- Classification Report



In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t.to(device))
    test_probs = torch.softmax(test_outputs, dim=1)
    test_preds = test_outputs.argmax(dim=1).cpu().numpy()

y_pred = test_preds
y_pred_prob = test_probs[:, 1].cpu().numpy()
y_true = y_test

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
test_auc = auc(fpr, tpr)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"  Accuracy:  {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  AUC-ROC:   {test_auc:.4f}")



In [ ]:
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Truth', 'Deception']))

cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(cm)



---
## Step 15: Visualize Results

We create comprehensive visualizations of training history and model performance.



In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Model Training & Evaluation Results", fontsize=16, fontweight='bold')

axes[0, 0].plot(history['train_loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', color='#e74c3c', linewidth=2)
axes[0, 0].set_title('Training & Validation Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['train_acc'], label='Train Acc', color='#3498db', linewidth=2)
axes[0, 1].plot(history['val_acc'], label='Val Acc', color='#e74c3c', linewidth=2)
axes[0, 1].set_title('Training & Validation Accuracy', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning rate curve
lrs = []
for epoch in range(len(history['train_loss'])):
    lrs.append(optimizer.param_groups[0]['lr'])
axes[0, 2].plot(lrs, color='#9b59b6', linewidth=2)
axes[0, 2].set_title('Learning Rate Schedule', fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Learning Rate')
axes[0, 2].set_yscale('log')
axes[0, 2].grid(True, alpha=0.3)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Truth', 'Deception'],
            yticklabels=['Truth', 'Deception'],
            linewidths=2, linecolor='black')
axes[1, 0].set_title('Confusion Matrix', fontweight='bold')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')

axes[1, 1].plot(fpr, tpr, color='#e74c3c', linewidth=2,
                label=f'ROC (AUC = {test_auc:.3f})')
axes[1, 1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[1, 1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1, 1].set_title('ROC Curve', fontweight='bold')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_prob)
axes[1, 2].plot(recall_curve, precision_curve, color='#2ecc71', linewidth=2)
axes[1, 2].set_title('Precision-Recall Curve', fontweight='bold')
axes[1, 2].set_xlabel('Recall')
axes[1, 2].set_ylabel('Precision')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_results.png'), dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Sample Predictions on Test Set", fontsize=14, fontweight='bold')

random_indices = random.sample(range(len(X_test)), min(10, len(X_test)))

for i, idx in enumerate(random_indices):
    row, col = divmod(i, 5)
    axes[row, col].imshow(X_test[idx])
    pred_label = 'Deception' if y_pred[idx] == 1 else 'Truth'
    true_label = 'Deception' if y_true[idx] == 1 else 'Truth'
    confidence = y_pred_prob[idx] if y_pred[idx] == 1 else 1 - y_pred_prob[idx]

    color = 'green' if y_pred[idx] == y_true[idx] else 'red'
    axes[row, col].set_title(
        f"Pred: {pred_label} ({confidence:.2f})\nTrue: {true_label}",
        fontweight='bold', color=color, fontsize=10
    )
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()



---
## Step 16: Save Model

We save the trained model and all artifacts for later use:
- Model weights (`.pth` format)
- Training history (`.json`)
- Label mappings
- Preprocessing configuration



In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, 'deception_detection_cnn.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'model_class': 'DeceptionCNN',
    'input_channels': 3,
    'num_classes': 2,
    'input_shape': list(X_train_t.shape[1:]),
}, model_path)
print(f"Model saved to: {model_path}")

history_path = os.path.join(MODEL_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to: {history_path}")

metadata = {
    'label_map': label_map,
    'input_shape': list(X_train_t.shape[1:]),
    'target_size': list(TARGET_SIZE),
    'epochs_trained': len(history['train_loss']),
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(test_acc),
    'test_auc': float(test_auc),
    'device': str(device),
}
meta_path = os.path.join(MODEL_DIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to: {meta_path}")

print(f"\nAll artifacts saved in: {MODEL_DIR}")



---
## Step 17: Predict on New Images

We load the saved model and predict on new, unseen images.
This demonstrates the complete inference pipeline:
1. Load model weights
2. Read and preprocess the input image (resize, normalize, transpose to CHW)
3. Run prediction
4. Output result with confidence score



In [ ]:
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
loaded_model = DeceptionCNN(
    input_channels=checkpoint['input_channels'],
    num_classes=checkpoint['num_classes']
).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()
print(f"Model loaded from: {model_path}")

with torch.no_grad():
    test_out = loaded_model(X_test_t.to(device))
    test_pred_loaded = test_out.argmax(dim=1).cpu().numpy()
loaded_acc = accuracy_score(y_true, test_pred_loaded)
print(f"Loaded model test accuracy: {loaded_acc:.4f}")



In [ ]:
def predict_frame(model, frame, target_size=(128, 128)):
    # frame: numpy (H, W, 3) RGB uint8
    resized = cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)
    enhanced = apply_clahe(resized)
    normalized = enhanced.astype(np.float32) / 255.0

    # HWC -> CHW -> tensor
    tensor = torch.tensor(normalized.transpose(2, 0, 1), dtype=torch.float32).unsqueeze(0)
    tensor = tensor.to(device)

    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)

    prob_deception = probs[0, 1].item()
    label = 'deception' if prob_deception > 0.5 else 'truth'
    confidence = prob_deception if prob_deception > 0.5 else 1 - prob_deception

    return label, confidence, prob_deception

print("Running predictions on test frames...\n")

for i in range(5):
    idx = random.randint(0, len(X_test) - 1)
    display_frame = (X_test[idx] * 255).astype(np.uint8)

    pred_label, confidence, raw_prob = predict_frame(loaded_model, all_frames[idx])
    true_label = 'deception' if y_true[idx] == 1 else 'truth'

    print(f"Sample {i+1}:")
    print(f"  Predicted: {pred_label.upper()} (confidence: {confidence:.2%})")
    print(f"  True:      {true_label.upper()}")
    print(f"  Raw prob:  {raw_prob:.4f}")
    print(f"  Correct:   {'YES' if pred_label == true_label else 'NO'}")
    print()



In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle("Final Predictions on Unseen Test Data", fontsize=15, fontweight='bold')

for i in range(10):
    idx = random.randint(0, len(X_test) - 1)
    row, col = divmod(i, 5)

    display_frame = (X_test[idx] * 255).astype(np.uint8)
    axes[row, col].imshow(display_frame)

    pred_label, confidence, _ = predict_frame(loaded_model, all_frames[idx])
    true_label = 'Deception' if y_true[idx] == 1 else 'Truth'

    is_correct = (pred_label.lower() == true_label.lower())
    title_color = 'green' if is_correct else 'red'
    check = 'V' if is_correct else 'X'

    axes[row, col].set_title(
        f"[{check}] Pred: {pred_label} ({confidence:.0%})\nTrue: {true_label}",
        fontweight='bold', color=title_color, fontsize=10
    )
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()



---
## Summary

### Completed Pipeline

| Step | Description | Status |
|------|-------------|--------|
| 1 | Import Libraries | Done |
| 2 | Load / Prepare Dataset | Done |
| 3 | Exploratory Data Analysis | Done |
| 4 | Data Cleaning | Done |
| 5 | Image Enhancement (CLAHE) | Done |
| 6 | Color-Space Processing (RGB, Gray, HSV, Edges) | Done |
| 7 | Resizing (128x128) | Done |
| 8 | Normalization (0-1) | Done |
| 9 | Data Augmentation (torchvision) | Done |
| 10 | Train/Validation/Test Split | Done |
| 11 | Build CNN (PyTorch) | Done |
| 12 | Compile Model (Adam + CrossEntropy) | Done |
| 13 | Train Model | Done |
| 14 | Evaluate Model | Done |
| 15 | Visualize Results | Done |
| 16 | Save Model (.pth) | Done |
| 17 | Predict on New Images | Done |

### Key Results
- **Test Accuracy**: See evaluation above
- **Model saved**: `saved_models/deception_detection_cnn.pth`
- **All outputs**: `cv_output/` directory

### Next Steps
- Increase dataset size (use all 1427 videos, not just 50)
- Try transfer learning (ResNet18, VGG16 from torchvision)
- Add face landmark detection
- Experiment with video-level aggregation (temporal models)

